<a href="https://colab.research.google.com/github/jenitchadha/SDS-510/blob/main/MOD5_BASICS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MODULE 5 BASICS PROJECT

In [20]:
### downloading the jeopardy dataset
import html
from urllib.request import urlretrieve

url = "https://www.dropbox.com/s/x7z8bk4vgowohp9/jeopardy.json?dl=1"
urlretrieve(url, "jeopardy.json")

('jeopardy.json', <http.client.HTTPMessage at 0x7a73aa50e8f0>)

In [16]:
## creating a df
import pandas as pd

df = pd.read_json("jeopardy.json")

print(f"## of rows: {len(df):,}")
df[["question", "value", "round"]].head()

## of rows: 216,930


,question,value,round
0,"'For the last 8 years of his life, Galileo was...",$200,Jeopardy!
1,'No. 2: 1912 Olympian; football star at Carlis...,$200,Jeopardy!
2,'The city of Yuma in this state has a record a...,$200,Jeopardy!
3,"'In 1963, live on ""The Art Linkletter Show"", t...",$200,Jeopardy!
4,"'Signer of the Dec. of Indep., framer of the C...",$200,Jeopardy!


In [19]:
## clearing up the dollar values under "value"
## here I will be adding a new column called "points"
## to store my processed data
df["points"] = pd.to_numeric(df["value"]
    .astype("string")
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip(),
    errors="coerce")

df = df.dropna(subset=["question", "points"])
df = df.loc[df["points"] > 0].copy()
df["points"] = df["points"].astype(int)

print(f"Rows after cleaning: {len(df):,}")
df[["question", "value", "points"]].head()

Rows after cleaning: 213,296


,question,value,points
0,"'For the last 8 years of his life, Galileo was...",$200,200
1,'No. 2: 1912 Olympian; football star at Carlis...,$200,200
2,'The city of Yuma in this state has a record a...,$200,200
3,"'In 1963, live on ""The Art Linkletter Show"", t...",$200,200
4,"'Signer of the Dec. of Indep., framer of the C...",$200,200


In [21]:
## now I will be doing something similar to the question
## and remove html tags and links, convert all the text into
## lowercase, and remove any extra spaces
## I will create a new "clean_question" column

df["clean_question"] = (df["question"]
    .map(html.unescape)
    .str.replace(r"<[^>]+>", " ", regex=True)
    .str.replace(r"https?://\S+", " ", regex=True)
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip())

df = df.loc[df["clean_question"].str.contains(r"\w{2,}")].copy()

print(f"Rows after cleaning the text: {len(df):,}")
df[["question", "clean_question"]].head()

Rows after cleaning the text: 213,235


,question,clean_question
0,"'For the last 8 years of his life, Galileo was...","'for the last 8 years of his life, galileo was..."
1,'No. 2: 1912 Olympian; football star at Carlis...,'no. 2: 1912 olympian; football star at carlis...
2,'The city of Yuma in this state has a record a...,'the city of yuma in this state has a record a...
3,"'In 1963, live on ""The Art Linkletter Show"", t...","'in 1963, live on ""the art linkletter show"", t..."
4,"'Signer of the Dec. of Indep., framer of the C...","'signer of the dec. of indep., framer of the c..."


In [22]:
### So we need to create a binary classifier with
### high and low values based on the points in each row.
### I am using a cut off of $600.

cutoff = 600

df["label"] = (df["points"] > cutoff).map({
    True: "high value",
    False: "low value"})

labels_per_clue = df.groupby("clean_question")["label"].transform("nunique")
df = df.loc[labels_per_clue == 1].copy()

df = df.drop_duplicates(subset="clean_question")

print(df["label"].value_counts())
print(f"Total clues to be split: {len(df):,}")

label
low value     119010
high value     92929
Name: count, dtype: int64
Total clues to be split: 211,939


In [23]:
## splitting training and test sets
from sklearn.model_selection import train_test_split

X = df["clean_question"]  # Clue text the model reads
y = df["label"]           # Correct label for each clue

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y)

print(f"Training set: {len(X_train):,}")
print(f"Test set: {len(X_test):,}")

Training set: 169,551
Test set: 42,388


In [24]:
## training the naive bayes classifier
## Naive Bayes learns which words are more common in each
## label using the training clues, then predicts a label
## for each test clue.
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline

model = make_pipeline(
    CountVectorizer(),
    MultinomialNB())

model.fit(X_train, y_train)
predictions = model.predict(X_test)

print(f"Predictions made: {len(predictions):,}")
print("First five predictions:", predictions[:5])

Predictions made: 42,388
First five predictions: ['low value' 'low value' 'high value' 'low value' 'low value']


In [25]:
## calculating accuracy

from pathlib import Path
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, predictions)

most_common_label = y_train.value_counts().idxmax()
baseline_accuracy = (y_test == most_common_label).mean()

results = (
    f"Test clues: {len(y_test):,}\n"
    f"Naive Bayes accuracy: {accuracy:.2%}\n"
    f"Baseline accuracy (always predict '{most_common_label}'): "
    f"{baseline_accuracy:.2%}\n"
    f"Difference: {(accuracy - baseline_accuracy) * 100:+.2f} percentage points\n")

print(results)
Path("jeopardy_results.txt").write_text(results, encoding="utf-8")

Test clues: 42,388
Naive Bayes accuracy: 58.07%
Baseline accuracy (always predict 'low value'): 56.15%
Difference: +1.92 percentage points



139

The model got 58.07% of the test clues right. Guessing “low value” every time got 56.15% right. Thus, the words in the clues helped the model a little.